In [ ]:
import os as _ka_os2
WORKING_DIR = "/kaggle/working" if _ka_os2.path.exists("/kaggle/working") else _ka_os2.getcwd()
SCRATCH_DIR = "/kaggle/tmp" if _ka_os2.path.exists("/kaggle/tmp") else _ka_os2.getcwd()


In [ ]:
import sys as _ka_sys
try:
    _ka_sys.stdout.reconfigure(line_buffering=True)
    _ka_sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass


In [ ]:
import os as _ka_os
def _gsv(name, default=None, cast=None):
    v = globals().get(name)
    if v is not None:
        return v
    e = _ka_os.environ.get(name)
    if e is None or not str(e).strip():
        return default
    try:
        return cast(e) if cast else e
    except (TypeError, ValueError):
        return default
SHARD_ID = _gsv('SHARD_ID', 0, cast=int)
TOTAL_SHARDS = _gsv('TOTAL_SHARDS', 1, cast=int)
print(f"shard={SHARD_ID}/{TOTAL_SHARDS}", flush=True)


In [ ]:
%%capture
import os, re
import torch
_tm = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__))
_tv = _tm.group(0) if _tm else "2.8"
xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(_tv, "0.0.34")
!uv pip install --system sentencepiece protobuf datasets huggingface_hub hf_transfer
!uv pip install --system --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!uv pip install --system --no-deps --upgrade torchao
!uv pip install --system --upgrade --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!uv pip install --system torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;


In [ ]:
import torch
print(f"gpus={torch.cuda.device_count()}", flush=True)
assert torch.cuda.device_count() > 0, "need T4 x2"
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", dtype=None, max_seq_length=1024, load_in_4bit=True, full_finetuning=False)
print("model loaded", flush=True)


In [ ]:
import json
REG = [{"video_id": "XId8oORl0iE", "center_start": 1057.0, "window": [{"start": 1053.28, "text": "டிசைட் பண்ணிக்கோங்க. சரியா? அடுத்து"}, {"start": 1055.36, "text": "வந்து பாத்தீங்கன்னா சிவகாசி விஜய்"}, {"start": 1056.96, "text": "அப்படிங்கற நண்பர் வந்து Samsung S25 எட்"}, {"start": 1059.28, "text": "60,000க்கு வாங்கலாமான்னு கேட்டுருக்காரு."}, {"start": 1060.799, "text": "இதுல புரிதல் ரொம்ப முக்கியம். நம்ம ஏன்"}], "silver": {"commenter_tamil": ["சிவகாசி விஜய்"], "commenter_english": ["sivakasivijay"], "mentioned": [], "note": "compound handle in prev chunk"}}, {"video_id": "h97k-dSt_98", "center_start": 0.1, "window": [{"start": 0.08, "text": "விசித்திரமான"}, {"start": 2.08, "text": "லுக்கா இருக்குங்க ஏரியாவே."}, {"start": 4.799, "text": "நீங்க அப்படியே பாத்தீங்கன்னு வைங்க."}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": [], "note": "scenery, no Q&A"}}, {"video_id": "bHp-SDnsnS8", "center_start": 1399.0, "window": [{"start": 1395.28, "text": "ஓடிறாங்க. ரைட் சோ அடுத்து வந்து"}, {"start": 1397.919, "text": "பாத்தீங்கன்னா டார்க் ஹன்டர் அப்படிங்கற"}, {"start": 1399.039, "text": "நண்பர் வந்து ஹாய் சிவான் கௌதம் ப்ரோ."}, {"start": 1400.48, "text": "பிக்ஸ் you மன்ஷன் did not work for me"}, {"start": 1402.48, "text": "என்னன்னா அந்த ஹாட்ஸ்ப ஆமா"}], "silver": {"commenter_tamil": ["டார்க் ஹன்டர்"], "commenter_english": ["darkhunter"], "mentioned": [], "note": "common-word compound"}}, {"video_id": "bHp-SDnsnS8", "center_start": 699.4, "window": [{"start": 696.32, "text": "மாட்டிட்டேன்."}, {"start": 697.12, "text": ">> எனக்கு₹ 15000 லாபம்."}, {"start": 699.44, "text": ">> வினோத் அப்படிங்கற நண்பர் வந்து ஹாய் ப்ரோ"}, {"start": 701.36, "text": "என் பிரண்ட் iPhone 16 pro மக்ஸ்"}, {"start": 702.88, "text": "வச்சிருக்கான். அந்த போன் யூஸ் பண்ணதுக்கு"}], "silver": {"commenter_tamil": ["வினோத்"], "commenter_english": ["vinoth"], "mentioned": [], "note": "same-chunk good case"}}, {"video_id": "XId8oORl0iE", "center_start": 774.2, "window": [{"start": 770.959, "text": "கண்டிப்பா நான் சந்திக்கிறேன். அடுத்து"}, {"start": 772.56, "text": "Infinix போன் நம்பி வாங்கலாமா"}, {"start": 774.24, "text": "அப்படின்னும்போது கீழ ஒரு நண்பர் வந்து"}, {"start": 776.16, "text": "Redmi விட Infinix ோட ஆஃப்டர் சேல்"}, {"start": 778.0, "text": "சப்போர்ட் நல்லா இருக்கு அப்படின்னு"}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": [], "note": "no handle, just 'oru nanbar'"}}, {"video_id": "XId8oORl0iE", "center_start": 697.8, "window": [{"start": 694.0, "text": "கட்டம் ஆயிடுமோ அப்படிங்கற மாதிரி சும்மா."}, {"start": 695.92, "text": "பட் இதுல ஹைலைட் என்னன்னா SK அப்படிங்கிற"}, {"start": 697.839, "text": "நண்பர் வந்து என்ன சொல்லிருக்காருன்னா"}, {"start": 699.12, "text": "கோயம்புத்தூர்ல VI 5G பறக்குது 325 MBS"}, {"start": 702.48, "text": "அப்படின்னு சொல்லி அனுப்பிருந்தாரு. சோ"}], "silver": {"commenter_tamil": ["SK"], "commenter_english": ["sk"], "mentioned": [], "note": "initials handle in prev chunk"}}, {"video_id": "bHp-SDnsnS8", "center_start": 390.0, "window": [{"start": 388.479, "text": "அதையும் நம்ம கன்சிடர் பண்ணலாம்."}, {"start": 389.52, "text": ">> கரெக்ட்."}, {"start": 390.0, "text": ">> அடுத்து ப்ரோ NHR 6259 அப்படிங்கற நண்பர்"}, {"start": 392.319, "text": "வந்து ஐம் யூசிங் Xiaomi 15"}, {"start": 394.0, "text": ">> அல்டிமேட் கேமரா. லிட்டில் டிராபேக்"}], "silver": {"commenter_tamil": ["NHR 6259"], "commenter_english": ["nhr6259"], "mentioned": [], "note": "digit handle read aloud"}}, {"video_id": "bHp-SDnsnS8", "center_start": 491.0, "window": [{"start": 487.199, "text": ">> அது போன் எல்லாருமே கேப்பாங்க. என்னடா"}, {"start": 489.28, "text": "கண்ணாடியே"}, {"start": 490.96, "text": ">> ஆமாவேல் பொன்ராஜ் அப்படிங்கற நண்பர் வந்து"}, {"start": 493.68, "text": "ஹாய் சிவா ப்ரோ. ஐம் யூசிங் S2 பாஸ்ட்"}, {"start": 496.08, "text": "3அப் இயர்ஸ் சரி வாங்க நம்ம போயிட்டு"}], "silver": {"commenter_tamil": ["பொன்ராஜ்"], "commenter_english": ["ponraj"], "mentioned": [], "note": "same-chunk good case"}}, {"video_id": "DJsfWQc9xWg", "center_start": 1926.2, "window": [{"start": 1923.44, "text": ">> உங்களுக்கு அந்த குவாலிட்டி இருக்குல்ல"}, {"start": 1924.88, "text": "கொடுக்குறோம்ல"}, {"start": 1926.159, "text": ">> அஸ்வத் ப்ரோ வந்துருக்கணும். சீரம்ல என்ன"}, {"start": 1928.0, "text": "நேரத்துக்கு விட்டு குவாலிட்டி"}, {"start": 1928.799, "text": "எடுத்துருப்பாரு."}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": ["அஸ்வத்"], "note": "crew, not commenter"}}, {"video_id": "XId8oORl0iE", "center_start": 290.5, "window": [{"start": 286.4, "text": "அப்டேட்ஸ் எவ்வளோ செக்யூரிட்டி பேட்ச்."}, {"start": 288.08, "text": "தேங்க்யூ ப்ரோ. நல்ல டேட்டா. அடுத்து"}, {"start": 290.479, "text": "கவின் என்ன சொல்லிருக்காருன்னா iPhone 16"}, {"start": 292.24, "text": "Pro வாங்கலாம்னு இருக்கேன் ப்ரோ. பிபோர்"}, {"start": 294.16, "text": "iPhone 18 Pro லான்ச் ஒர்த்தா இருக்குமா"}], "silver": {"commenter_tamil": ["கவின்"], "commenter_english": ["kavin"], "mentioned": [], "note": "same-chunk good case"}}, {"video_id": "XId8oORl0iE", "center_start": 914.2, "window": [{"start": 909.12, "text": "ஓகே ரைட் சோ அடுத்து வந்து பாத்தீங்கன்னா"}, {"start": 912.8, "text": "ஓகே இங்கயும் பாத்தீங்கன்னா விக்னேஷ்"}, {"start": 914.16, "text": "விக்கி அப்படிங்கற நண்பரும் கய்ஸ் எனக்கு"}, {"start": 916.48, "text": "ஹெல்ப் பண்ணுங்க டிசிஷன் எடுக்கணும். நான்"}, {"start": 919.04, "text": "வந்து நத்திங் போன் 3 வாங்கலாமா இல்லாட்டி"}], "silver": {"commenter_tamil": ["விக்கி"], "commenter_english": ["vikki"], "mentioned": [], "note": "same-chunk good case"}}, {"video_id": "DJsfWQc9xWg", "center_start": 1202.5, "window": [{"start": 1199.84, "text": "ஐயா மாதவ"}, {"start": 1201.12, "text": ">> ஆனா அந்த அளவுக்குல்லாம் இவர் பண்ணலங்க"}, {"start": 1202.48, "text": "மாதவ அங்கிள் வந்து நல்லாவே இந்த போன"}, {"start": 1204.08, "text": "பண்ணிருக்காரு."}, {"start": 1204.559, "text": ">> அதாவது ரீபிராண்டிங் அதனால என்ன"}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": ["மாதவ்"], "note": "story person, not commenter"}}, {"video_id": "XId8oORl0iE", "center_start": 124.0, "window": [{"start": 120.32, "text": "நம்ம வந்து என்ன பண்ணுவோம் சோஃர்ஸ்ட்"}, {"start": 122.32, "text": "என்னன்னா கரூஸ் அண்டர்ஸ கண்ட்ரோல்"}, {"start": 124.0, "text": "அப்படிங்கற நண்பர் வந்து யடாக்அபட் os"}, {"start": 126.479, "text": "அண்ட் செக்யூரிட்டி அப்டேட் someம் ஆப்ஸ்வ"}, {"start": 128.399, "text": "run afterட afterட yearயஸ் இif itட் isஸ்"}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": [], "note": "garbled handle, needs human listen"}}, {"video_id": "XId8oORl0iE", "center_start": 943.0, "window": [{"start": 940.079, "text": "உங்களுக்கு 35 கிடைக்கலாம் இல்லையா சோ"}, {"start": 941.76, "text": "அந்த அடிப்படையில இந்த சந்தேகத்தை"}, {"start": 943.04, "text": "கேக்குறாரு. இல்லன்னா வந்து நான் நத்திங்"}, {"start": 944.72, "text": "போன் 3 வாங்கட்டுமா பிரைஸ் ஹைக் வந்து"}, {"start": 946.56, "text": "எனக்கு கொஞ்சம் பயமுடுத்தது அப்படின்னு"}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": [], "note": "handle 2+ chunks earlier, tests window width"}}, {"video_id": "DJsfWQc9xWg", "center_start": 977.2, "window": [{"start": 973.519, "text": ">> ஆமா ஜிடி வந்து ஆல்ரெடி மேலதான்"}, {"start": 975.759, "text": "இருக்காங்க. அவங்க மேலயே இருந்துட்டோம்"}, {"start": 977.199, "text": ">> இல்ல சாய் சுதர்சனும் கில்லும்"}, {"start": 978.56, "text": "இறங்குனாங்கனாலே முடிஞ்சது ரெண்டு பேரும்."}, {"start": 980.48, "text": ">> என் மாப்பிள்ளை வந்து இந்த சீசன சாமிக்கு"}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": ["சாய் சுதர்சன்", "கில்"], "note": "cricketers, not commenters"}}, {"video_id": "bHp-SDnsnS8", "center_start": 819.0, "window": [{"start": 812.32, "text": "இருந்துச்சு எனக்கு என்ன இது"}, {"start": 816.16, "text": "என்னன்னா சதீஷ் துரைசாமி அப்படிங்கற"}, {"start": 819.04, "text": "நண்பர் வந்து டியர் சிவாபுரன் கௌதம் ப்ரோஐ"}, {"start": 821.839, "text": "wouldட்லை to explain the vodafone"}, {"start": 823.279, "text": "reality in ukே"}], "silver": {"commenter_tamil": ["சதீஷ் துரைசாமி"], "commenter_english": ["sathishduraisamy"], "mentioned": ["சிவாபுரன் கௌதம்"], "note": "addressee trap: SBUS himself in body"}}, {"video_id": "DJsfWQc9xWg", "center_start": 222.0, "window": [{"start": 218.72, "text": ">> நாங்களே இன்னைக்கு போடுற வீடியோவே நாங்க"}, {"start": 220.959, "text": "என்னன்னு மறந்துருவோம். அடுத்த நாள்"}, {"start": 222.0, "text": "கேட்டுிட்டு இருப்போம் பரத் ப்ரோ"}, {"start": 222.879, "text": "கேப்பாரு. சிவா அந்த வீடியோ போட்டியா"}, {"start": 224.799, "text": "அப்படின்னு."}], "silver": {"commenter_tamil": [], "commenter_english": [], "mentioned": ["பரத்"], "note": "co-host, not commenter"}}]
print(f"regression rows={len(REG)}", flush=True)
assert len(REG) == 17
SYS = ('Identify the commenter whose comment is read aloud in this Tamil transcript window. Reply JSON only: {"commenter_tamil": [...], "commenter_english": [...], "mentioned": [...]}. Empty lists if none.\n'
'Rules: the commenter handle appears BEFORE cue words like \u0b85\u0baa\u0bcd\u0baa\u0b9f\u0bbf\u0b99\u0bcd\u0b95\u0bb1/\u0b85\u0baa\u0bcd\u0baa\u0b9f\u0bbf\u0b99\u0bcd\u0b95\u0bbf\u0bb1 \u0ba8\u0ba3\u0bcd\u0baa\u0bb0\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bc1, \u0b95\u0bc7\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bbe\u0bb0\u0bc1. Extract that span.\n'
'Keep the WHOLE handle verbatim even if compound: brand+name, place+name, English words, initials, digits. Never truncate to one part. Keep digits.\n'
'commenter_english is the most likely original handle spelling, lowercase, no spaces. mentioned holds names AFTER the cue (players, crew, the host) - never put them in commenter.\n'
'Never output tech words alone, greetings, verbs, or common words (iphone, processor, snapdragon, ram, bro, hello, guys).\n'
'Only extract spans verbatim in the Input. Never copy names from these examples.\n'
'Example: Input: \u0bb5\u0ba8\u0bcd\u0ba4\u0bc1 \u0baa\u0bbe\u0ba4\u0bcd\u0ba4\u0bc0\u0b99\u0bcd\u0b95\u0ba9\u0bcd\u0ba9\u0bbe \u0b9a\u0bbf\u0bb5\u0b95\u0bbe\u0b9a\u0bbf \u0bb5\u0bbf\u0b9c\u0baf\u0bcd / \u0b85\u0baa\u0bcd\u0baa\u0b9f\u0bbf\u0b99\u0bcd\u0b95\u0bb1 \u0ba8\u0ba3\u0bcd\u0baa\u0bb0\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bc1 Samsung S25 Output: {"commenter_tamil": ["\u0b9a\u0bbf\u0bb5\u0b95\u0bbe\u0b9a\u0bbf \u0bb5\u0bbf\u0b9c\u0baf\u0bcd"], "commenter_english": ["sivakasivijay"], "mentioned": []}\n'
'Example: Input: \u0b85\u0b9f\u0bc1\u0ba4\u0bcd\u0ba4\u0bc1 \u0baa\u0bcd\u0bb0\u0bcb NHR 6259 \u0b85\u0baa\u0bcd\u0baa\u0b9f\u0bbf\u0b99\u0bcd\u0b95\u0bb1 \u0ba8\u0ba3\u0bcd\u0baa\u0bb0\u0bcd / \u0bb5\u0ba8\u0bcd\u0ba4\u0bc1 \u0b90\u0bae\u0bcd \u0baf\u0bc2\u0b9a\u0bbf\u0b99\u0bcd Xiaomi 15 Output: {"commenter_tamil": ["NHR 6259"], "commenter_english": ["nhr6259"], "mentioned": []}\n'
'Example: Input: \u0b9a\u0ba4\u0bc0\u0bb7\u0bcd \u0ba4\u0bc1\u0bb0\u0bc8\u0b9a\u0bbe\u0bae\u0bbf \u0b85\u0baa\u0bcd\u0baa\u0b9f\u0bbf\u0b99\u0bcd\u0b95\u0bb1 / \u0ba8\u0ba3\u0bcd\u0baa\u0bb0\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bc1 \u0b9f\u0bbf\u0baf\u0bb0\u0bcd \u0b9a\u0bbf\u0bb5\u0bbe\u0baa\u0bc1\u0bb0\u0ba9\u0bcd \u0b95\u0bcc\u0ba4\u0bae\u0bcd \u0baa\u0bcd\u0bb0\u0bcb Output: {"commenter_tamil": ["\u0b9a\u0ba4\u0bc0\u0bb7\u0bcd \u0ba4\u0bc1\u0bb0\u0bc8\u0b9a\u0bbe\u0bae\u0bbf"], "commenter_english": ["sathishduraisamy"], "mentioned": ["\u0b9a\u0bbf\u0bb5\u0bbe\u0baa\u0bc1\u0bb0\u0ba9\u0bcd \u0b95\u0bcc\u0ba4\u0bae\u0bcd"]}\n'
'Example: Input: \u0b87\u0bb2\u0bcd\u0bb2 \u0b9a\u0bbe\u0baf\u0bcd \u0b9a\u0bc1\u0ba4\u0bb0\u0bcd\u0b9a\u0ba9\u0bc1\u0bae\u0bcd \u0b95\u0bbf\u0bb2\u0bcd\u0bb2\u0bc1\u0bae\u0bcd / \u0b87\u0bb1\u0b99\u0bcd\u0b95\u0bc1\u0ba9\u0bbe\u0b99\u0bcd\u0b95\u0ba9\u0bbe\u0bb2\u0bc7 Output: {"commenter_tamil": [], "commenter_english": [], "mentioned": ["\u0b9a\u0bbe\u0baf\u0bcd \u0b9a\u0bc1\u0ba4\u0bb0\u0bcd\u0b9a\u0ba9\u0bcd", "\u0b95\u0bbf\u0bb2\u0bcd"]}\n'
'Example: Input: \u0baa\u0bcd\u0bb0\u0bcb iphone processor \u0ba8\u0bb2\u0bcd\u0bb2\u0bbe \u0b87\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1. Output: {"commenter_tamil": [], "commenter_english": [], "mentioned": []}\n'
'Input: {TEXT}\nOutput:')
def build_prompt(row):
    window_text = " / ".join(w["text"] for w in row["window"])
    return SYS.replace("{TEXT}", window_text)


In [ ]:
import json
def norm_eng(s):
    return "".join(ch for ch in s.lower() if ch.isalnum())
def score_rows(rows, preds):
    agg = {"n": 0, "commenter_exact": 0, "english_match": 0, "exp_handles": 0, "leak": 0, "fp": 0, "clean_empty": 0, "empty_total": 0}
    details = []
    for r, p in zip(rows, preds):
        s = r["silver"]
        exp_c, exp_e, exp_m = s["commenter_tamil"], s["commenter_english"], s["mentioned"]
        got_c = p.get("commenter_tamil", [])
        got_e = p.get("commenter_english", [])
        has_exp = bool(exp_c)
        exact = got_c == exp_c
        eng = [norm_eng(x) for x in got_e] == [norm_eng(x) for x in exp_e]
        leak = bool(set(got_c) & set(exp_m))
        agg["n"] += 1
        agg["commenter_exact"] += exact
        if has_exp:
            agg["exp_handles"] += 1
            agg["english_match"] += eng
        else:
            agg["empty_total"] += 1
            agg["clean_empty"] += not got_c
            agg["fp"] += bool(got_c) and not leak
        agg["leak"] += leak
        details.append({"video_id": r["video_id"], "center": r["center_start"], "exact": exact, "eng": eng, "leak": leak, "pred": p, "note": s["note"]})
    return agg, details


In [ ]:
import json, time
OUTPUT_PATH = WORKING_DIR + f"/pilot_handles_v2_shard{SHARD_ID}.jsonl"
def llm_extract(text):
    prompt = SYS.replace("{TEXT}", text)
    msgs = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    try:
        _dev = model.language_model.embed_tokens.weight.device
    except Exception:
        _dev = "cuda:0"
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(_dev)
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    cleaned = gen.replace("json", "").replace("```", "").strip()
    try:
        p = json.loads(cleaned)
        return {"commenter_tamil": p.get("commenter_tamil", []), "commenter_english": p.get("commenter_english", []), "mentioned": p.get("mentioned", [])}, False
    except Exception:
        return {"commenter_tamil": [], "commenter_english": [], "mentioned": []}, True
t0 = time.perf_counter()
preds, fails = [], 0
out = open(OUTPUT_PATH, "w", encoding="utf-8")
for ri, r in enumerate(REG):
    wtext = " / ".join(w["text"] for w in r["window"])
    p, f = llm_extract(wtext)
    fails += f
    preds.append(p)
    json.dump({"video_id": r["video_id"], "center_start": r["center_start"], "pred": p, "silver": r["silver"]}, out, ensure_ascii=False)
    out.write("\n")
    print(f"{ri + 1}/{len(REG)} fail={f} pred={json.dumps(p, ensure_ascii=False)}", flush=True)
out.close()
agg, details = score_rows(REG, preds)
agg["fails"] = fails
agg["secs"] = round(time.perf_counter() - t0, 1)
print(json.dumps(agg, indent=1, ensure_ascii=False), flush=True)
with open(WORKING_DIR + "/pilot_handles_v2_metrics.json", "w", encoding="utf-8") as mf:
    json.dump({"agg": agg, "details": details}, mf, ensure_ascii=False, indent=1)
print(f"saved {OUTPUT_PATH}", flush=True)


In [ ]:
import gc, shutil, os
for _n in ["model", "tokenizer"]:
    try:
        del globals()[_n]
    except KeyError:
        pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
for _p in [os.path.expanduser("~/.cache/huggingface"), os.path.expanduser("~/.cache/uv"), "/root/.cache/huggingface"]:
    try:
        if os.path.isdir(_p):
            shutil.rmtree(_p, ignore_errors=True)
            print(f"removed {_p}", flush=True)
    except Exception as _e:
        print(f"cleanup skip {_p}: {_e}", flush=True)
print("model files + caches removed; outputs kept", flush=True)
